# Azure AI Search — Provisionamento dos índices da workshop

Este notebook constrói **dois índices** na instância `ai-search-ai-framework`
a partir dos dados sintéticos em `data/indexsampledata/`:

| Índice                | Origem dos dados                                  |
|-----------------------|----------------------------------------------------|
| `telecom-products`    | `data/indexsampledata/telecom-products/*.json`    |
| `internal-regulations`| `data/indexsampledata/internal-regulations/*.json`|

Os dois índices usam **vetorização integrada via Azure OpenAI**: a Search Service
chama `text-embedding-3-small` no Foundry usando sua *system-assigned managed
identity* (sem chaves).

**Pré-requisitos** (mesmas instruções do `example.env`):
- `az login` com identidade que tenha `Search Service Contributor` + `Search Index Data Contributor`.
- AI Search com identidade gerenciada e role `Cognitive Services OpenAI User` no Foundry.
- Foundry com deployment `text-embedding-3-small`.

In [1]:
%pip install --quiet --upgrade \
    "azure-search-documents>=11.5.1" \
    "azure-identity>=1.17.0" \
    "openai>=1.40.0" \
    "python-dotenv>=1.0.1"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

NOTEBOOK_DIR = Path.cwd()
ENV_PATH = NOTEBOOK_DIR / ".env"
if not ENV_PATH.exists():
    raise FileNotFoundError(f".env not found at {ENV_PATH}. Copy example.env to .env and fill in values.")
load_dotenv(ENV_PATH, override=True)

SEARCH_ENDPOINT       = os.environ["AZURE_SEARCH_ENDPOINT"]
INDEX_TELECOM         = os.environ.get("AZURE_SEARCH_INDEX_TELECOM", "telecom-products")
INDEX_REGULATIONS     = os.environ.get("AZURE_SEARCH_INDEX_REGULATIONS", "internal-regulations")

AOAI_ENDPOINT         = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
AOAI_API_VERSION      = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21")
AOAI_EMBED_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
AOAI_EMBED_MODEL      = os.environ.get("AZURE_OPENAI_EMBEDDING_MODEL", AOAI_EMBED_DEPLOYMENT)
AOAI_EMBED_DIMENSIONS = int(os.environ.get("AZURE_OPENAI_EMBEDDING_DIMENSIONS", "1536"))

DATA_DIR_TELECOM      = (NOTEBOOK_DIR / os.environ.get("SAMPLE_DATA_DIR_TELECOM", "../../data/indexsampledata/telecom-products")).resolve()
DATA_DIR_REGULATIONS  = (NOTEBOOK_DIR / os.environ.get("SAMPLE_DATA_DIR_REGULATIONS", "../../data/indexsampledata/internal-regulations")).resolve()

credential = DefaultAzureCredential()

print(f"Search endpoint   : {SEARCH_ENDPOINT}")
print(f"Telecom index     : {INDEX_TELECOM} <- {DATA_DIR_TELECOM}")
print(f"Regulations index : {INDEX_REGULATIONS} <- {DATA_DIR_REGULATIONS}")
print(f"AOAI embed deploy : {AOAI_EMBED_DEPLOYMENT} ({AOAI_EMBED_DIMENSIONS} dims)")

Search endpoint   : https://ai-search-ai-framework.search.windows.net
Telecom index     : telecom-products <- C:\Users\ptonpauletti\framework-ai-contoso\data\indexsampledata\telecom-products
Regulations index : internal-regulations <- C:\Users\ptonpauletti\framework-ai-contoso\data\indexsampledata\internal-regulations
AOAI embed deploy : text-embedding-3-small (1536 dims)


## 1. Helpers genéricos: schema builder, embeddings e upload em lote

Cada índice tem campos diferentes, mas o pipeline (criar schema → ler JSON →
embeddar → fazer upload) é igual. Os helpers abaixo isolam essa lógica.

In [3]:
import json, hashlib
from typing import Callable, Iterable
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, VectorSearchProfile, HnswAlgorithmConfiguration,
    AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters,
    SemanticConfiguration, SemanticSearch, SemanticPrioritizedFields, SemanticField,
)
from azure.identity import get_bearer_token_provider
from openai import AzureOpenAI

VECTOR_PROFILE  = "vp-aoai"
VECTOR_ALGO     = "hnsw-default"
VECTORIZER_NAME = "vec-aoai"
SEMANTIC_CONFIG = "sem-default"

aoai_token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
aoai_client = AzureOpenAI(
    azure_endpoint=AOAI_ENDPOINT,
    azure_ad_token_provider=aoai_token_provider,
    api_version=AOAI_API_VERSION,
)

def make_id(*parts: str) -> str:
    raw = "|".join(p.lower() for p in parts if p)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

def vector_search_block() -> VectorSearch:
    return VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name=VECTOR_ALGO)],
        profiles=[VectorSearchProfile(
            name=VECTOR_PROFILE,
            algorithm_configuration_name=VECTOR_ALGO,
            vectorizer_name=VECTORIZER_NAME,
        )],
        vectorizers=[AzureOpenAIVectorizer(
            vectorizer_name=VECTORIZER_NAME,
            parameters=AzureOpenAIVectorizerParameters(
                resource_url=AOAI_ENDPOINT,
                deployment_name=AOAI_EMBED_DEPLOYMENT,
                model_name=AOAI_EMBED_MODEL,
                # No api_key -> Search MI calls AOAI
            ),
        )],
    )

def upsert_index(index_name: str, fields: list, semantic_block: SemanticSearch) -> None:
    client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential, api_version="2025-11-01-preview")
    existing = {idx.name for idx in client.list_indexes()}
    if index_name in existing:
        print(f"[{index_name}] já existe -> recriando")
        client.delete_index(index_name)
    idx = SearchIndex(
        name=index_name,
        fields=fields,
        vector_search=vector_search_block(),
        semantic_search=semantic_block,
    )
    client.create_index(idx)
    print(f"[{index_name}] criado ({len(fields)} campos)")

def embed_batch(texts: list[str]) -> list[list[float]]:
    resp = aoai_client.embeddings.create(model=AOAI_EMBED_DEPLOYMENT, input=texts)
    return [item.embedding for item in sorted(resp.data, key=lambda x: x.index)]

def upload_documents(index_name: str, docs: list[dict], vector_field: str, source_field: str) -> None:
    EMBED_BATCH = 16
    for start in range(0, len(docs), EMBED_BATCH):
        chunk = docs[start:start + EMBED_BATCH]
        vectors = embed_batch([d[source_field] for d in chunk])
        for d, v in zip(chunk, vectors):
            d[vector_field] = v
        print(f"  [{index_name}] embedded {start + len(chunk)}/{len(docs)}")
    sc = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=index_name, credential=credential, api_version="2025-11-01-preview")
    UPLOAD_BATCH = 100
    total = 0
    for start in range(0, len(docs), UPLOAD_BATCH):
        batch = docs[start:start + UPLOAD_BATCH]
        results = sc.merge_or_upload_documents(documents=batch)
        ok = sum(1 for r in results if r.succeeded)
        total += ok
        print(f"  [{index_name}] uploaded {ok}/{len(batch)}")
    print(f"[{index_name}] total uploaded: {total}/{len(docs)}")

## 2. Índice `telecom-products`

Schema otimizado para catálogo de planos móveis, fibra/TV e dispositivos.
O campo `groupField` agrega o texto descritivo que será embeddado.

In [4]:
telecom_fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
    SearchableField(name="sku", type=SearchFieldDataType.String, filterable=True, sortable=True),
    SearchableField(name="productName", type=SearchFieldDataType.String, filterable=True, sortable=True),
    SearchableField(name="category", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchableField(name="subcategory", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="monthlyPriceBRL", type=SearchFieldDataType.Double, filterable=True, sortable=True, facetable=True),
    SimpleField(name="loyaltyMonths", type=SearchFieldDataType.Int32, filterable=True, facetable=True),
    SimpleField(name="dataAllowanceGB", type=SearchFieldDataType.Int32, filterable=True, sortable=True, facetable=True),
    SimpleField(name="downloadMbps", type=SearchFieldDataType.Int32, filterable=True, sortable=True),
    SimpleField(name="uploadMbps", type=SearchFieldDataType.Int32, filterable=True, sortable=True),
    SimpleField(name="compatible5G", type=SearchFieldDataType.Boolean, filterable=True, facetable=True),
    SearchableField(name="targetSegment", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchableField(name="manufacturer", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchableField(name="description", type=SearchFieldDataType.String),
    SearchableField(name="groupField", type=SearchFieldDataType.String),
    SearchField(
        name="groupFieldVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=AOAI_EMBED_DIMENSIONS,
        vector_search_profile_name=VECTOR_PROFILE,
    ),
]

telecom_semantic = SemanticSearch(
    default_configuration_name=SEMANTIC_CONFIG,
    configurations=[SemanticConfiguration(
        name=SEMANTIC_CONFIG,
        prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="productName"),
            content_fields=[
                SemanticField(field_name="groupField"),
                SemanticField(field_name="description"),
            ],
            keywords_fields=[
                SemanticField(field_name="category"),
                SemanticField(field_name="subcategory"),
                SemanticField(field_name="targetSegment"),
            ],
        ),
    )],
)

upsert_index(INDEX_TELECOM, telecom_fields, telecom_semantic)

[telecom-products] já existe -> recriando


[telecom-products] criado (16 campos)


In [5]:
def build_telecom_group(rec: dict) -> str:
    parts = [
        rec.get("productName", ""),
        rec.get("category", ""),
        rec.get("subcategory", ""),
        rec.get("targetSegment", ""),
        rec.get("manufacturer", ""),
        rec.get("description", ""),
    ]
    extras = []
    if rec.get("monthlyPriceBRL"):
        extras.append(f"R${rec['monthlyPriceBRL']:.2f}/mês")
    if rec.get("dataAllowanceGB"):
        extras.append(f"{rec['dataAllowanceGB']}GB")
    if rec.get("downloadMbps"):
        extras.append(f"{rec['downloadMbps']}Mbps download / {rec.get('uploadMbps',0)}Mbps upload")
    if rec.get("compatible5G"):
        extras.append("5G")
    return " | ".join([p for p in parts if p] + extras)

telecom_docs: list[dict] = []
for fp in sorted(DATA_DIR_TELECOM.glob("*.json")):
    payload = json.loads(fp.read_text(encoding="utf-8"))
    for wrapper in payload:
        for rec in wrapper.get("products", []):
            doc = {
                "id": make_id(rec.get("sku", ""), rec.get("productName", "")),
                "sku": rec.get("sku", ""),
                "productName": rec.get("productName", ""),
                "category": rec.get("category", ""),
                "subcategory": rec.get("subcategory", ""),
                "monthlyPriceBRL": float(rec.get("monthlyPriceBRL") or 0.0),
                "loyaltyMonths": int(rec.get("loyaltyMonths") or 0),
                "dataAllowanceGB": int(rec.get("dataAllowanceGB") or 0),
                "downloadMbps": int(rec.get("downloadMbps") or 0),
                "uploadMbps": int(rec.get("uploadMbps") or 0),
                "compatible5G": bool(rec.get("compatible5G")),
                "targetSegment": rec.get("targetSegment", ""),
                "manufacturer": rec.get("manufacturer", ""),
                "description": rec.get("description", ""),
            }
            doc["groupField"] = build_telecom_group(rec)
            telecom_docs.append(doc)

print(f"Loaded {len(telecom_docs)} telecom products from {DATA_DIR_TELECOM}")
upload_documents(INDEX_TELECOM, telecom_docs, vector_field="groupFieldVector", source_field="groupField")

Loaded 30 telecom products from C:\Users\ptonpauletti\framework-ai-contoso\data\indexsampledata\telecom-products


  [telecom-products] embedded 16/30


  [telecom-products] embedded 30/30


  [telecom-products] uploaded 30/30
[telecom-products] total uploaded: 30/30


## 3. Índice `internal-regulations`

Schema voltado a documentos de política interna. Cada registro vira um chunk único
(política inteira) — para um cenário real, particionar por seção seria melhor.

In [6]:
reg_fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
    SearchableField(name="policyId", type=SearchFieldDataType.String, filterable=True, sortable=True),
    SearchableField(name="title", type=SearchFieldDataType.String, filterable=True, sortable=True),
    SearchableField(name="category", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchableField(name="subcategory", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="version", type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="effectiveDate", type=SearchFieldDataType.String, filterable=True, sortable=True),
    SearchableField(name="owner", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchableField(name="audience", type=SearchFieldDataType.String, filterable=True),
    SearchableField(name="summary", type=SearchFieldDataType.String),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchableField(name="groupField", type=SearchFieldDataType.String),
    SearchField(
        name="groupFieldVector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=AOAI_EMBED_DIMENSIONS,
        vector_search_profile_name=VECTOR_PROFILE,
    ),
]

reg_semantic = SemanticSearch(
    default_configuration_name=SEMANTIC_CONFIG,
    configurations=[SemanticConfiguration(
        name=SEMANTIC_CONFIG,
        prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="title"),
            content_fields=[
                SemanticField(field_name="content"),
                SemanticField(field_name="summary"),
            ],
            keywords_fields=[
                SemanticField(field_name="category"),
                SemanticField(field_name="subcategory"),
                SemanticField(field_name="owner"),
            ],
        ),
    )],
)

upsert_index(INDEX_REGULATIONS, reg_fields, reg_semantic)

[internal-regulations] criado (13 campos)


In [7]:
def build_reg_group(rec: dict) -> str:
    return " | ".join([
        rec.get("policyId", ""),
        rec.get("title", ""),
        rec.get("category", ""),
        rec.get("subcategory", ""),
        rec.get("audience", ""),
        rec.get("summary", ""),
        rec.get("content", ""),
    ])

reg_docs: list[dict] = []
for fp in sorted(DATA_DIR_REGULATIONS.glob("*.json")):
    payload = json.loads(fp.read_text(encoding="utf-8"))
    for wrapper in payload:
        for rec in wrapper.get("regulations", []):
            doc = {
                "id": make_id(rec.get("policyId", ""), rec.get("title", "")),
                "policyId": rec.get("policyId", ""),
                "title": rec.get("title", ""),
                "category": rec.get("category", ""),
                "subcategory": rec.get("subcategory", ""),
                "version": rec.get("version", ""),
                "effectiveDate": rec.get("effectiveDate", ""),
                "owner": rec.get("owner", ""),
                "audience": rec.get("audience", ""),
                "summary": rec.get("summary", ""),
                "content": rec.get("content", ""),
            }
            doc["groupField"] = build_reg_group(rec)
            reg_docs.append(doc)

print(f"Loaded {len(reg_docs)} regulations from {DATA_DIR_REGULATIONS}")
upload_documents(INDEX_REGULATIONS, reg_docs, vector_field="groupFieldVector", source_field="groupField")

Loaded 21 regulations from C:\Users\ptonpauletti\framework-ai-contoso\data\indexsampledata\internal-regulations


  [internal-regulations] embedded 16/21


  [internal-regulations] embedded 21/21


  [internal-regulations] uploaded 21/21
[internal-regulations] total uploaded: 21/21


## 4. Smoke tests

Hybrid search nos dois índices, usando o vetorizador integrado.

In [8]:
from azure.search.documents.models import VectorizableTextQuery

sc_t = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=INDEX_TELECOM, credential=credential, api_version="2025-11-01-preview")
q = "plano familia 4 linhas com 5G e Spotify"
hits = sc_t.search(
    search_text=q,
    vector_queries=[VectorizableTextQuery(text=q, k_nearest_neighbors=5, fields="groupFieldVector")],
    select=["sku", "productName", "category", "subcategory", "monthlyPriceBRL"],
    top=5,
)
print(f"Top 5 — {q!r}")
for h in hits:
    print(f"  [{h['@search.score']:.3f}] {h['productName']} ({h['subcategory']}) R${h['monthlyPriceBRL']:.2f}")

Top 5 — 'plano familia 4 linhas com 5G e Spotify'


  [0.033] ConectaTel Família 4 linhas 160GB (Família) R$269.90
  [0.033] ConectaTel Família 2 linhas 80GB (Família) R$149.90
  [0.031] ConectaTel Empresa 5 linhas 300GB (PME) R$449.90
  [0.031] ConectaTel Pós Plus 50GB (Pós-pago) R$119.90
  [0.031] ConectaTel Combo Família Fibra 700 + 2 chips (Combo) R$199.90


In [9]:
sc_r = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=INDEX_REGULATIONS, credential=credential, api_version="2025-11-01-preview")
q = "posso usar ChatGPT pessoal para revisar contrato de cliente?"
hits = sc_r.search(
    search_text=q,
    vector_queries=[VectorizableTextQuery(text=q, k_nearest_neighbors=5, fields="groupFieldVector")],
    select=["policyId", "title", "category", "subcategory", "owner"],
    top=5,
)
print(f"Top 5 — {q!r}")
for h in hits:
    print(f"  [{h['@search.score']:.3f}] {h['policyId']} {h['title']} ({h['owner']})")

Top 5 — 'posso usar ChatGPT pessoal para revisar contrato de cliente?'


  [0.033] SI-006 Política de Uso de Inteligência Artificial Generativa (CISO + Comitê de IA Responsável)
  [0.032] LGPD-003 Política de Compartilhamento e Transferência Internacional (DPO + Jurídico)
  [0.031] LGPD-006 Política de Tratamento de Dados de Crianças e Adolescentes (DPO)
  [0.031] LGPD-001 Política Geral de Proteção de Dados Pessoais (DPO)
  [0.030] LGPD-004 Política de Retenção e Descarte de Dados (DPO)
